<a href="https://colab.research.google.com/github/Minakshi654/DocSense--CNN-based-Document-Classifier/blob/main/05_lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers peft accelerate bitsandbytes datasets -q

import torch
print("GPU available:", torch.cuda.is_available())
print("GPU memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB" if torch.cuda.is_available() else "")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.2 MB/s eta 0:00:00
GPU available: True
GPU memory: 15.637086208 GB


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default, reuse end-of-sequence token

base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model = base_model.to("cuda")

print(base_model.config)
print("\nTotal parameters:", sum(p.numel() for p in base_model.parameters()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.10.2",
  "use_cache": true,
  "vocab_size": 50257
}


Total parameters: 12443980

In [3]:
def generate_text(model, prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = "Summarize the following retail sales report in 2-3 sentences:\n\nRevenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market.\n\nSummary:"

print(generate_text(base_model, prompt))

Summarize the following retail sales report in 2-3 sentences:

Revenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market.

Summary:

Total revenue grew 4.8% in Q4 compared with the previous year.

Revenue grew 4.8% in Q4 compared with the previous year. The UK had the highest average retail sales of any major trading country, followed by the United States and Ireland.

Revenue grew 4.8% in Q4 compared with the previous year. The UK had the


In [4]:
training_examples = [
    {
        "report": "Revenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market, generating over 25 times the revenue of the next closest country.",
        "summary": "Q4 revenue rose sharply due to holiday demand, with the UK continuing to dominate as the primary market."
    },
    {
        "report": "Average order value was £480.87, but median order value was only £303.04, indicating the average is skewed by a small number of large bulk orders.",
        "summary": "Order values are skewed by bulk purchases; median (£303.04) is more representative than the average (£480.87)."
    },
    {
        "report": "February recorded the lowest monthly revenue of the year at £447,137, while November peaked at £1,161,817, more than double February's figure.",
        "summary": "Revenue ranged widely across the year, from a February low of £447K to a November peak of £1.16M."
    },
    {
        "report": "Home decor and giftware products dominated the top 10 best-selling items, suggesting customers primarily shop for gifts and household items rather than bulk commodities.",
        "summary": "Top sellers were dominated by home decor and gift items, pointing to a gift-and-household-focused customer base."
    },
    {
        "report": "The Netherlands, Ireland, Germany, and France each generated over £200,000 in revenue, representing the strongest international markets outside the UK.",
        "summary": "Outside the UK, the Netherlands, Ireland, Germany, and France emerged as the strongest secondary markets."
    },
    {
        "report": "135,080 transactions were excluded due to missing customer identifiers, representing nearly a quarter of the original dataset before cleaning.",
        "summary": "Roughly 25% of transactions were excluded due to missing customer data, highlighting a data quality gap."
    },
    {
        "report": "Sales of seasonal items spiked sharply in the weeks leading up to major holidays, then dropped off quickly once the holiday period ended.",
        "summary": "Seasonal product sales spiked before holidays and fell off quickly afterward."
    },
    {
        "report": "Customer retention rates were highest among buyers who made a second purchase within 30 days of their first order.",
        "summary": "Early repeat purchases within 30 days strongly predicted higher customer retention."
    },
]

print("Training examples:", len(training_examples))
print("\nExample 1:")
print("Report:", training_examples[0]["report"])
print("Summary:", training_examples[0]["summary"])

Training examples: 8

Example 1:
Report: Revenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market, generating over 25 times the revenue of the next closest country.
Summary: Q4 revenue rose sharply due to holiday demand, with the UK continuing to dominate as the primary market.


In [5]:
def format_example(example):
    return f"Report: {example['report']}\nSummary: {example['summary']}{tokenizer.eos_token}"

formatted_texts = [format_example(ex) for ex in training_examples]

print(formatted_texts[0])
print("\n---\n")
print(formatted_texts[1])

Report: Revenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market, generating over 25 times the revenue of the next closest country.
Summary: Q4 revenue rose sharply due to holiday demand, with the UK continuing to dominate as the primary market.<|endoftext|>

---

Report: Average order value was £480.87, but median order value was only £303.04, indicating the average is skewed by a small number of large bulk orders.
Summary: Order values are skewed by bulk purchases; median (£303.04) is more representative than the average (£480.87).<|endoftext|>


In [6]:
from datasets import Dataset

def tokenize_function(examples):
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128  # short, since our examples are brief
    )
    tokenized["labels"] = tokenized["input_ids"].copy()  # for language modeling, labels = input_ids
    return tokenized

train_dataset = Dataset.from_dict({"text": formatted_texts})
tokenized_dataset = train_dataset.map(tokenize_function, batched=True)

print(tokenized_dataset)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 8
})


In [7]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,   # we're doing text generation, not classification
    r=8,                             # rank of the adapter matrices - controls their size/capacity
    lora_alpha=16,                   # scaling factor for the adapter's influence
    lora_dropout=0.1,                # dropout for regularization, same concept as our CNN
    target_modules=["c_attn"],       # which layers inside GPT-2 to attach adapters to (its attention layers)
)

lora_model = get_peft_model(base_model, lora_config)

lora_model.print_trainable_parameters()

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [8]:
!pip install -U torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.0 MB/s eta 0:00:00


In [9]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lora_results",
    num_train_epochs=20,           # small dataset needs more passes to learn the pattern
    per_device_train_batch_size=2,
    learning_rate=2e-4,            # LoRA typically uses a higher learning rate than full fine-tuning
    logging_steps=5,
    save_strategy="no",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

print("Trainer ready")

Trainer ready


In [11]:
import datasets
datasets.config.TORCHVISION_AVAILABLE = False

trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,8.508759
10,8.286873
15,7.877378
20,7.674979
25,7.080936
30,6.468391
35,5.792070
40,5.377460
45,4.772634
50,4.360967


TrainOutput(global_step=80, training_loss=5.334008455276489, metrics={'train_runtime': 7.4884, 'train_samples_per_second': 21.366, 'train_steps_per_second': 10.683, 'total_flos': 10487920066560.0, 'train_loss': 5.334008455276489, 'epoch': 20.0})

In [12]:
lora_model = get_peft_model(base_model, lora_config)

training_args = TrainingArguments(
    output_dir="./lora_results",
    num_train_epochs=60,           # significantly more passes over our tiny dataset
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="no",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
10,8.370834
20,7.584183
30,6.226300
40,4.623970
50,2.809999
60,1.852504
70,1.683644
80,1.537714
90,1.521874
100,1.468271


TrainOutput(global_step=240, training_loss=2.317369695504506, metrics={'train_runtime': 16.1764, 'train_samples_per_second': 29.673, 'train_steps_per_second': 14.836, 'total_flos': 31463760199680.0, 'train_loss': 2.317369695504506, 'epoch': 60.0})

In [13]:
lora_model.eval()

prompt = "Report: Revenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market.\nSummary:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = lora_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

print("LoRA fine-tuned output:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

LoRA fine-tuned output:
Report: Revenue grew significantly in Q4, driven by strong holiday demand. The UK remained the dominant market.
Summary: Q4 was a strong performer with strong holiday demand.


In [14]:
prompt2 = "Report: Customer complaints decreased by a notable margin after the new support system was introduced, improving overall satisfaction scores.\nSummary:"

inputs2 = tokenizer(prompt2, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs2 = lora_model.generate(
        **inputs2,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

print("LoRA output on new topic:")
print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

LoRA output on new topic:
Report: Customer complaints decreased by a notable margin after the new support system was introduced, improving overall satisfaction scores.
Summary: Customer complaints experienced a decrease in quality as well as increased quality, while customer complaints were higher than expected.


In [15]:
lora_model.save_pretrained("gpt2-lora-summarizer")
tokenizer.save_pretrained("gpt2-lora-summarizer")

!zip -r gpt2-lora-summarizer.zip gpt2-lora-summarizer

from google.colab import files
files.download("gpt2-lora-summarizer.zip")

print("Saved and ready for download")

  adding: gpt2-lora-summarizer/ (stored 0%)
  adding: gpt2-lora-summarizer/tokenizer_config.json (deflated 50%)
  adding: gpt2-lora-summarizer/README.md (deflated 66%)
  adding: gpt2-lora-summarizer/tokenizer.json (deflated 82%)
  adding: gpt2-lora-summarizer/adapter_config.json (deflated 59%)
  adding: gpt2-lora-summarizer/adapter_model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved and ready for download
